In [2]:
!pip install skl2onnx onnxmltools onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 58.7 MB/s eta 0:00:00


In [5]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

In [4]:
data_path = "/content/telco-customer-churn.csv"

In [6]:
# Safety check for Colab directory structure
if not os.path.exists("onnx-exports"):
    os.makedirs("onnx-exports")

In [8]:
df = pd.read_csv(data_path)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

In [9]:
# 2. Features
numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
                        "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                        "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
                        "PaperlessBilling", "PaymentMethod"]

X = df[numeric_features + categorical_features]
y = df["Churn"].apply(lambda x: 1 if x == "Yes" else 0)

In [10]:
# 3. Split 70/30 (Matching your Java Demo)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
# 4. Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', LogisticRegression(max_iter=1000))])

In [12]:
# 5. Train & Evaluate
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n=== LAB EVALUATION (PYTHON) ===")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))


=== LAB EVALUATION (PYTHON) ===
              precision    recall  f1-score   support

    No Churn       0.85      0.90      0.88      1539
       Churn       0.69      0.57      0.62       574

    accuracy                           0.81      2113
   macro avg       0.77      0.74      0.75      2113
weighted avg       0.81      0.81      0.81      2113



In [13]:
# 6. Export to ONNX
# This is where we define the 'float_input' and 'string_input' for Java
initial_types = (
    [(col, FloatTensorType([None, 1])) for col in numeric_features] +
    [(col, StringTensorType([None, 1])) for col in categorical_features]
)

onx = convert_sklearn(clf, initial_types=initial_types)
with open("onnx-exports/churn_model.onnx", "wb") as f:
    f.write(onx.SerializeToString())